# 💈 Barber & Hairdresser Lead Finder — 50 miles of Wigan
**No install needed. Just click the ▶ buttons top to bottom, one at a time.**

At the end a spreadsheet will automatically download to your computer.

In [ ]:
# ── Step 1: Install packages (click ▶ and wait ~10 seconds) ──────────────
!pip install -q requests beautifulsoup4 openpyxl lxml
print('✅ Packages ready')

In [ ]:
# ── Step 2: Run the lead finder (takes 5–15 minutes) ─────────────────────
import re, math, time, unicodedata
from datetime import datetime
from urllib.parse import urljoin
import requests, openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from bs4 import BeautifulSoup

WIGAN_LAT  = 53.5450
WIGAN_LNG  = -2.6325
RADIUS_MI  = 50
RADIUS_M   = int(RADIUS_MI * 1609.344)
OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
YELL_BASE    = 'https://www.yell.com/ucs/UcsSearchAction.do'
YELL_KEYWORDS  = ['barbers', 'hairdressers', 'hair salon', 'barbershop']
YELL_MAX_PAGES = 20
OUTDATED_YEARS = 3
SOCIAL_DOMAINS = ('facebook.com','instagram.com','twitter.com','tiktok.com','linkedin.com','linktree.com')
BROWSER_HEADERS = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}
YELL_HEADERS = {**BROWSER_HEADERS, 'Accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8','Accept-Language':'en-GB,en;q=0.9','Referer':'https://www.yell.com/'}

def haversine_mi(lat1,lng1,lat2,lng2):
    R=3958.8; p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lng2-lng1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def _osm_address(tags):
    parts=[tags.get('addr:housenumber',''),tags.get('addr:street',''),tags.get('addr:city','') or tags.get('addr:town',''),tags.get('addr:postcode','')]
    return ', '.join(p for p in parts if p)

def fetch_from_overpass():
    query=f"""[out:json][timeout:90];
(
  node["shop"="hairdresser"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
  way["shop"="hairdresser"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
  node["shop"="barber"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
  way["shop"="barber"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
  node["amenity"="hairdresser"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
  way["amenity"="hairdresser"](around:{RADIUS_M},{WIGAN_LAT},{WIGAN_LNG});
);
out center tags;""".strip()
    print('Querying OpenStreetMap...', end=' ', flush=True)
    try:
        resp=requests.post(OVERPASS_URL,data={'data':query},headers={'User-Agent':'WiganLeadFinder/1.0'},timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f'Error: {e}'); return []
    elements=resp.json().get('elements',[])
    print(f'{len(elements)} elements found.')
    results=[]
    for el in elements:
        tags=el.get('tags',{}); name=tags.get('name','').strip()
        if not name: continue
        if el['type']=='node': blat,blng=el.get('lat'),el.get('lon')
        else:
            c=el.get('center',{}); blat,blng=c.get('lat'),c.get('lon')
        if blat is None: continue
        dist=haversine_mi(WIGAN_LAT,WIGAN_LNG,blat,blng)
        if dist>RADIUS_MI: continue
        results.append({'name':name,'phone':tags.get('phone') or tags.get('contact:phone',''),'website':tags.get('website') or tags.get('contact:website',''),'address':_osm_address(tags),'rating':'','reviews':0,'distance_mi':round(dist,1),'lat':blat,'lng':blng,'source':'osm'})
    print(f'OpenStreetMap: {len(results)} businesses within {RADIUS_MI} miles.')
    return results

def _fetch_yell_page(keyword,page_num):
    try:
        resp=requests.get(YELL_BASE,params={'keywords':keyword,'location':'Wigan, Greater Manchester','radius':'50','pageNum':str(page_num)},headers=YELL_HEADERS,timeout=20)
        if resp.status_code==200: return resp.text
    except: pass
    return ''

def _parse_yell_page(html):
    soup=BeautifulSoup(html,'lxml')
    articles=soup.find_all('article',class_=lambda c:c and 'businessCapsule--listing' in c)
    results=[]
    for art in articles:
        def _t(cls): tag=art.find(class_=lambda c:c and cls in c); return tag.get_text(strip=True) if tag else ''
        name=_t('businessCapsule--name')
        if not name: continue
        site_tag=art.find('a',class_=lambda c:c and 'businessCapsule--website' in c)
        website=(site_tag.get('data-url') or site_tag.get('href','')) if site_tag else ''
        dist_txt=_t('businessCapsule--distance')
        dist_mi=0.0
        m=re.search(r'([\d.]+)\s*miles?',dist_txt,re.I)
        if m: dist_mi=float(m.group(1))
        results.append({'name':name,'phone':_t('businessCapsule--telephone'),'website':website,'address':_t('businessCapsule--address'),'rating':'','reviews':0,'distance_mi':dist_mi,'lat':None,'lng':None,'source':'yell'})
    return results

def fetch_from_yell():
    all_results=[]
    for keyword in YELL_KEYWORDS:
        print(f"Yell.com: '{keyword}' ",end='',flush=True)
        count=0
        for page_num in range(1,YELL_MAX_PAGES+1):
            html=_fetch_yell_page(keyword,page_num)
            if not html: break
            pr=_parse_yell_page(html)
            if not pr: break
            filtered=[r for r in pr if r['distance_mi']<=RADIUS_MI]
            all_results.extend(filtered); count+=len(filtered)
            print('.',end='',flush=True)
            if len(pr)<10: break
            time.sleep(1.5)
        print(f' {count}')
    print(f'Yell total: {len(all_results)} listings.')
    return all_results

_UK_PC_RE=re.compile(r'\b([A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2})\b',re.I)
def _norm(s): s=unicodedata.normalize('NFKD',s).encode('ascii','ignore').decode(); return re.sub(r'[^a-z0-9]','',s.lower())
def _postcode(addr): m=_UK_PC_RE.search(addr); return _norm(m.group(1)) if m else ''
def deduplicate(osm,yell):
    seen={}; merged=[]
    for biz in osm+yell:
        words=biz['name'].split(); first=_norm(words[0]) if words else ''
        pc=_postcode(biz['address']); key=(first,pc) if pc else (_norm(biz['name']),)
        if key not in seen: seen[key]=True; merged.append(biz)
    return merged

def check_website(url):
    if not url: return 'none','No website listed'
    if any(d in url.lower() for d in SOCIAL_DOMAINS):
        d=next(x for x in SOCIAL_DOMAINS if x in url.lower()); return 'social_only',f'Only a {d} page'
    try:
        resp=requests.get(url,headers=BROWSER_HEADERS,timeout=12,allow_redirects=True)
    except requests.exceptions.SSLError: return 'outdated','Broken SSL'
    except requests.exceptions.ConnectionError: return 'error','Cannot connect'
    except requests.exceptions.Timeout: return 'error','Timed out'
    except Exception as e: return 'error',str(e)[:80]
    if resp.status_code>=400: return 'error',f'HTTP {resp.status_code}'
    cur=datetime.now().year; text=resp.text
    lm=resp.headers.get('Last-Modified','')
    if lm:
        try:
            from email.utils import parsedate; p=parsedate(lm)
            if p and cur-p[0]>=OUTDATED_YEARS: return 'outdated',f'Last-Modified: {p[0]}'
        except: pass
    yrs=re.findall(r'copyright[^\d]{0,10}(\d{4})',text,re.I)+re.findall(r'[©]\s*(\d{4})',text)+re.findall(r'&copy;\s*(\d{4})',text)
    valid=[int(y) for y in yrs if 2000<=int(y)<=cur+1]
    if valid and cur-max(valid)>=OUTDATED_YEARS: return 'outdated',f'Copyright: {max(valid)}'
    soup=BeautifulSoup(text,'lxml')
    gen=soup.find('meta',{'name':'generator'})
    if gen:
        c=gen.get('content','').lower(); wp=re.search(r'wordpress\s+([\d.]+)',c)
        if wp:
            try:
                if float(wp.group(1))<5.0: return 'outdated',f'Old WordPress {wp.group(1)}'
            except: pass
    all_tags=soup.find_all()
    if all_tags and len(soup.find_all('table'))/len(all_tags)>0.12: return 'outdated','Table-based layout'
    return 'active','Website looks current'

EMAIL_RE=re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')
SKIP_EMAIL={'noreply','no-reply','example','test','wordpress','sentry','privacy','abuse','postmaster'}
WA_HREF_RE=re.compile(r'wa\.me/(\+?[\d]+)|whatsapp\.com/send\?phone=([\d+]+)',re.I)
WA_TEXT_RE=re.compile(r'whatsapp[\s:]*[\+]?[\d\s\-()]{9,18}',re.I)

def _get_html(url,timeout=10):
    try:
        r=requests.get(url,headers=BROWSER_HEADERS,timeout=timeout,allow_redirects=True)
        if r.status_code==200: return r.text
    except: pass
    return ''

def scrape_contacts(url):
    if not url or any(d in url.lower() for d in SOCIAL_DOMAINS): return '',''
    email=whatsapp=''
    for page_url in [url]+[urljoin(url,s) for s in ('/contact','/contact-us','/about')]:
        if email and whatsapp: break
        html=_get_html(page_url)
        if not html: continue
        soup=BeautifulSoup(html,'lxml')
        if not email:
            for tag in soup.find_all('a',href=re.compile(r'^mailto:',re.I)):
                m=re.search(r'mailto:([^\?&\s]+)',tag['href'])
                if m:
                    c=m.group(1).strip().lower()
                    if not any(s in c for s in SKIP_EMAIL): email=c; break
        if not email:
            for c in EMAIL_RE.findall(html):
                if not any(s in c.lower() for s in SKIP_EMAIL): email=c.lower(); break
        if not whatsapp:
            for tag in soup.find_all('a',href=True):
                m=WA_HREF_RE.search(tag['href'])
                if m: whatsapp=(m.group(1) or m.group(2)).strip(); break
        if not whatsapp:
            m=WA_TEXT_RE.search(html)
            if m:
                digits=re.findall(r'[\d+]{10,13}',m.group())
                if digits: whatsapp=digits[0]
    return email,whatsapp

COLS=[('Business Name',30),('Phone Number',18),('Email Address',32),('WhatsApp Number',18),('Website Status',16),('Website / Social URL',42),('Address',45),('Rating',10),('Reviews',10),('Notes',36),('Distance (miles)',16)]
FILL_HEADER=PatternFill('solid',fgColor='1A2035'); FILL_NONE=PatternFill('solid',fgColor='FFD6D6')
FILL_SOCIAL=PatternFill('solid',fgColor='D6EAFF'); FILL_OUTDATED=PatternFill('solid',fgColor='FFF2CC')
FILL_ERROR=PatternFill('solid',fgColor='E8E8E8'); FILL_ALT=PatternFill('solid',fgColor='F7F7F7')
STATUS_ORDER={'none':0,'social_only':1,'outdated':2,'error':3,'active':4}
THIN=Border(**{s:Side(style='thin',color='CCCCCC') for s in ('left','right','top','bottom')})
def _row_fill(status,idx): return {'none':FILL_NONE,'social_only':FILL_SOCIAL,'outdated':FILL_OUTDATED,'error':FILL_ERROR}.get(status,FILL_ALT if idx%2==0 else PatternFill(fill_type=None))

def save_spreadsheet(leads):
    ts=datetime.now().strftime('%Y%m%d_%H%M%S'); filename=f'wigan_hair_leads_{ts}.xlsx'
    wb=openpyxl.Workbook(); ws=wb.active; ws.title='Leads'
    for col,(hdr,width) in enumerate(COLS,1):
        c=ws.cell(1,col,hdr); c.fill=FILL_HEADER; c.font=Font(color='FFFFFF',bold=True,size=10)
        c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True); c.border=THIN
        ws.column_dimensions[get_column_letter(col)].width=width
    ws.row_dimensions[1].height=32
    sorted_leads=sorted(leads,key=lambda b:(STATUS_ORDER.get(b['status'],9),b['distance_mi']))
    for ri,biz in enumerate(sorted_leads,2):
        fill=_row_fill(biz['status'],ri)
        for col,val in enumerate([biz['name'],biz['phone'],biz['email'],biz['whatsapp'],biz['status'].upper().replace('_',' '),biz['website'],biz['address'],biz['rating'],biz['reviews'],biz['notes'],biz['distance_mi']],1):
            c=ws.cell(ri,col,val); c.fill=fill; c.alignment=Alignment(vertical='center'); c.border=THIN
        ws.row_dimensions[ri].height=18
    ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions
    ws2=wb.create_sheet('Summary')
    cnts={s:sum(1 for b in leads if b['status']==s) for s in ('none','social_only','outdated','error')}
    for r,(label,val) in enumerate([('Search area',f'{RADIUS_MI}-mile radius of Wigan, UK'),('Date',datetime.now().strftime('%d/%m/%Y %H:%M')),('Sources','OpenStreetMap + Yell.com'),('',''),('Total leads',len(leads)),('No website',cnts['none']),('Social only',cnts['social_only']),('Outdated',cnts['outdated']),('Unreachable',cnts['error']),('',''),('With email',sum(1 for b in leads if b['email'])),('With WhatsApp',sum(1 for b in leads if b['whatsapp'])),('With phone',sum(1 for b in leads if b['phone'])),('',''),('RED = No website','Best leads'),('BLUE = Social only','Facebook/Instagram only'),('AMBER = Outdated','Old or broken site'),('GREY = Unreachable','Site down')],1):
        ws2.cell(r,1,label).font=Font(bold=True,size=10); ws2.cell(r,2,val)
    ws2.column_dimensions['A'].width=26; ws2.column_dimensions['B'].width=40
    wb.save(filename); return filename,sorted_leads

# ── RUN ───────────────────────────────────────────────────────────────────
print('='*55)
print(' Lead Finder — Barbers & Hairdressers, 50mi of Wigan')
print('='*55)
print()
print('[1/3] Fetching businesses...')
osm_biz=fetch_from_overpass()
print()
yell_biz=fetch_from_yell()
raw=deduplicate(osm_biz,yell_biz)
print(f'\n{len(raw)} unique businesses ({len(osm_biz)} OSM + {len(yell_biz)} Yell, after dedup)')
print()
print('[2/3] Checking websites & scraping contacts...')
leads=[]
for idx,biz in enumerate(raw,1):
    status,notes=check_website(biz['website'])
    if idx%20==0: print(f'  {idx}/{len(raw)} checked...')
    if status=='active': continue
    email,whatsapp='',''
    if biz['website'] and status!='none': email,whatsapp=scrape_contacts(biz['website'])
    leads.append({**biz,'status':status,'notes':notes,'email':email,'whatsapp':whatsapp})
print(f'  {len(leads)} leads found (skipped {len(raw)-len(leads)} with active websites)')
print()
print('[3/3] Saving spreadsheet...')
filename,sorted_leads=save_spreadsheet(leads)
cnts={s:sum(1 for b in sorted_leads if b['status']==s) for s in ('none','social_only','outdated','error')}
print()
print('='*55)
print(f'  DONE!  {len(sorted_leads)} leads saved to {filename}')
print(f'  No website : {cnts["none"]}')
print(f'  Social only: {cnts["social_only"]}')
print(f'  Outdated   : {cnts["outdated"]}')
print(f'  With email : {sum(1 for b in sorted_leads if b["email"])}')
print('='*55)

In [ ]:
# ── Step 3: Download the spreadsheet to your computer ────────────────────
from google.colab import files
files.download(filename)
print('✅ Download started — check your Downloads folder')